In [1]:
import sys
from pathlib import Path

# Add project root to Python path for imports
project_root = Path.cwd().parent
common_path = project_root / "common"

# Add to sys.path only if not already added
if str(common_path) not in sys.path:
    sys.path.insert(0, str(common_path))

# OpenAI Agent - News Headlines

In [3]:
import os
import sys

from dotenv import load_dotenv
from agents import Agent, Runner, trace

from aagents.web_agent import web_agent

load_dotenv()

import json
from typing import Any, List, Dict
from IPython.display import display, Markdown

def format_search_results_md(results: Any) -> str:
    """
    Convert search-results JSON (string/dict/list) to a Markdown string.
    Expects each item to be a dict with keys like: title, link/url, snippet/body/summary, source.
    """
    # Normalize input
    if isinstance(results, str):
        try:
            results = json.loads(results)
        except json.JSONDecodeError:
            # Not JSON — return as a simple paragraph
            return f"```\n{results}\n```"

    # Drill into common wrapper
    if isinstance(results, dict):
        if "results" in results and isinstance(results["results"], list):
            items = results["results"]
        elif "items" in results and isinstance(results["items"], list):
            items = results["items"]
        else:
            # If dict looks like a single result, wrap it
            # Otherwise try to find a list value inside
            list_vals = [v for v in results.values() if isinstance(v, list)]
            items = list_vals[0] if list_vals else [results]
    elif isinstance(results, list):
        items = results
    else:
        return str(results)

    # Build markdown
    md_lines: List[str] = []
    md_lines.append(f"# Search Results ({len(items)})\n")

    for i, r in enumerate(items, start=1):
        # Handle Pydantic models
        if hasattr(r, "model_dump"):
            r = r.model_dump()
        elif hasattr(r, "dict"):
            r = r.dict()

        if not isinstance(r, dict):
            # If result item is primitive, just show it
            md_lines.append(f"## {i}. Result\n\n```\n{r}\n```\n")
            continue

        title = r.get("title") or r.get("heading") or r.get("name") or "No title"
        link = r.get("link") or r.get("url") or ""
        snippet = r.get("snippet") or r.get("body") or r.get("summary") or ""
        source = r.get("source") or r.get("site") or ""
        date = r.get("datetime") or r.get("date") or ""

        # Title line
        if date:
            md_lines.append(f"### {i}. {title} ({date})\n")
        else:
            md_lines.append(f"### {i}. {title}\n")
               

        # Metadata line
        meta_parts = []
        if link:
            meta_parts.append(f"**Link:** {link}")
        if source:
            meta_parts.append(f"**Source:** {source}")
        if meta_parts:
            md_lines.append(" · ".join(meta_parts) + "\n")

        # Snippet
        if snippet:
            # Clean up snippet
            snippet = snippet.replace("\n", " ").strip()
            md_lines.append(f"> {snippet}\n")
        
        md_lines.append("---\n") # Separator

    return "\n".join(md_lines)


async def run_test():

    with trace("News Headline Agent Run"):
        response = await Runner.run(
            web_agent,
            input=(
                "Find out RECENT news headlines in India? " \
                "" \
                "The news should be the MOST recent available. " \
                "Ignore anything older than 1 day."
            ),
        )


    print("[DEBUG] Agent Final Output:\n")
    print(format_search_results_md(response.final_output))

await run_test()

[DEBUG] duckduckgo_search called with: query='recent news headlines in India' max_results=5 search_type='news' timelimit='d' region='in-en'
[DEBUG] duckduckgo_search returning 3 results
[DEBUG] Agent Final Output:

# Search Results (3)

### 1. Apple Fitness+ launches in India on December 15: Price, features, and more (2025-10-11T15:46:00+00:00)

**Link:** https://www.msn.com/en-in/money/technology/apple-fitness-launches-in-india-on-december-15-price-features-and-more/ar-AA1RWrsb?ocid=BingNewsVerp

> Apple Fitness+ is launching in India in India on December 15. Priced at Rs 149 per month or Rs 999 annually, the ...

---

### 2. China’s latest provocation is pushing India too far (2025-10-20T15:46:00+00:00)

**Link:** https://www.msn.com/en-us/politics/international-relations/china-s-latest-provocation-is-pushing-india-too-far/vi-AA1RWAYd?ocid=BingNewsVerp

> Tensions between India and China are rising again after years of uneasy calm along their disputed border. This video explains ...


# OpenAI Agent - Performing Market Research

In [5]:
from aagents.web_research_agent import web_research_agent
from dotenv import load_dotenv
from agents import Runner, trace

load_dotenv()

async def run_test():
    with trace("Market Research Agent Run"):
        response = await Runner.run(
            web_research_agent,
            "Popular MLops tools in 2025."
        )

    print("[DEBUG] Agent Final Output:\n")
    print(response.final_output)

await run_test()


[DEBUG] duckduckgo_search called with: query='Popular MLOps tools in 2025' max_results=5 search_type='text' timelimit='m' region='us-en'
[DEBUG] duckduckgo_search returning 5 results
[DEBUG] fetch_page_content called with: https://www.braintrust.dev/articles/best-llmops-platforms-2025 - timeout: 3
[DEBUG] fetch_page_content called with: https://research.aimultiple.com/llmops-tools/ - timeout: 3
[DEBUG] fetch_page_content called with: https://aitoolinsight.com/best-ai-tools-2025/ - timeout: 3
[DEBUG] fetch_page_content called with: https://uplatz.com/blog/the-2025-mlops-landscape-a-comparative-analysis-of-mlflow-weights-biases-and-neptune/ - timeout: 3
[DEBUG] fetch_page_content called with: https://www.analyticsinsight.net/machine-learning/10-must-know-python-libraries-for-mlops-in-2025 - timeout: 3
[DEBUG] Agent Final Output:

### Popular MLOps Tools in 2025

As of 2025, the MLOps landscape has evolved to include a variety of platforms and tools specifically designed to streamline the

# OpenAI Agent - Market Sentiment Analysis

In [ ]:
import os
import sys
from aagents.web_research_agent import web_research_agent
from dotenv import load_dotenv
from agents import Runner, trace

load_dotenv()

async def run_test():
    with trace("Stock Market Sentiment Analysis Agent Run"):
        response = await Runner.run(
            web_research_agent,
            "Stock market sentiment analysis for major tech companies."
        )

    print("[DEBUG] Agent Final Output:\n")
    print(response.final_output)

await run_test()


[DEBUG] duckduckgo_search called with: query='Stock market sentiment analysis major tech companies' max_results=5 search_type='text' timelimit='d' region='us-en'
[DEBUG] duckduckgo_search returning 5 results
[DEBUG] fetch_page_content called with: https://www.marketbeat.com/stocks/top-rated-tech-stocks/ - timeout: 3
[WARNING] Failed to fetch content from https://www.marketbeat.com/stocks/top-rated-tech-stocks/: 403 Client Error: Forbidden for url: https://www.marketbeat.com/stocks/top-rated-tech-stocks/
[DEBUG] fetch_page_content called with: https://www.seeitmarket.com/stock-market-outlook-big-tech-economic-data-tropical-storms-and-tik-tok/ - timeout: 3
[WARNING] Failed to fetch content from https://www.seeitmarket.com/stock-market-outlook-big-tech-economic-data-tropical-storms-and-tik-tok/: 403 Client Error: Forbidden for url: https://www.seeitmarket.com/stock-market-outlook-big-tech-economic-data-tropical-storms-and-tik-tok/
[DEBUG] fetch_page_content called with: https://www.entrep

# OpenAI Agent - Reporting Weather Forecast

In [28]:
import os
import sys
from aagents.weather_agent import weather_agent
from dotenv import load_dotenv
from agents import Runner, trace

load_dotenv()

async def run_test():
    with trace("Weather Agent Run"):
        response = await Runner.run(
            weather_agent,
            "How is the weather in Melissa and Dallas for next 2 days? Today's date is 2025-12-07.",
        )

    print("[DEBUG] Agent Final Output:\n")
    print(response.final_output)

await run_test()


[DEBUG] Primary API get_weather_forecast called for city=Melissa
[DEBUG] Primary API get_weather_forecast called for city=Dallas
[DEBUG] Primary API get_weather_forecast called for city=Dallas
[DEBUG] Primary API get_weather_forecast called for city=Melissa
[DEBUG] Primary API get_weather_forecast called for city=Melissa
[DEBUG] Primary API get_weather_forecast called for city=Dallas
[DEBUG] Primary API get_weather_forecast called for city=Dallas
[DEBUG] Agent Final Output:

Here is the weather forecast for Melissa and Dallas over the next two days:

### Melissa Weather Forecast

#### December 7, 2025
- **Description:** Overcast clouds
- **Temperature:** 5.65°C
- **Humidity:** 90%
- **Wind Speed:** 7.01 m/s

#### December 8, 2025
- **Description:** Scattered clouds
- **Temperature:** 6.19°C
- **Humidity:** 61%
- **Wind Speed:** 6.78 m/s

---

### Dallas Weather Forecast

#### December 7, 2025
- **Description:** Overcast clouds
- **Temperature:** 7.18°C
- **Humidity:** 96%
- **Wind Spee